<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/Computer-Science/Operating-Systems/02-processes-and-program-execution.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Operating Systems guideline](Operating-Systems.html)


## **Processes and Program Execution**

A **process** is the operating system's managed instance of a program in execution. A program file contains instructions and data, but it has no live register values, no scheduled CPU time, no private virtual address space, and no kernel-maintained identity. Those properties appear only when the operating system creates an execution context around the program.

This distinction is central to almost every later operating-system topic. Scheduling chooses among processes or threads, virtual memory protects their address spaces, file systems expose resources through descriptors, and security checks use process credentials. A process is therefore more than "a program currently on the CPU." It remains a process while it is ready, sleeping, stopped, or waiting for a child, even though it is not executing instructions at that moment.

The running example for this chapter is the shell command:

```bash
cat input.txt | grep kernel > result.txt
```

What looks like one command line actually asks the shell and kernel to coordinate several mechanisms:

- create a pipe and open `result.txt`;
- create two child processes;
- redirect their standard descriptors;
- replace each child's program image with `cat` or `grep`;
- schedule the resulting processes independently;
- deliver end-of-file when every unused pipe writer has closed; and
- retain each exit status until the shell collects it.

Following that path gives a concrete answer to the chapter's main question: **how does a passive executable become a protected, observable, and reclaimable computation?**

### **Program, Process, and Execution Context**

The words *program* and *process* are sometimes used casually as synonyms, but the operating system must keep them separate. A program is reusable input. A process is a temporary execution that owns or references resources and whose state changes over time.

#### **A Program Is Not Yet a Process**

An executable file such as `/usr/bin/grep` is an organized sequence of bytes on persistent storage. Its headers describe how code and data should be mapped, where execution should begin, and whether a dynamic linker is required. The file can remain unchanged for years and can be executed by thousands of processes.

To run it, the operating system must combine at least four kinds of state:

| Kind of state | Typical contents | Why it is needed |
|---|---|---|
| Program representation | machine instructions, initialized data, loader metadata | Defines what computation can be performed |
| Memory context | page tables, mappings, stack, heap, shared libraries | Gives instructions and data protected virtual addresses |
| CPU context | program counter, stack pointer, general registers, status flags | Records exactly where execution can resume |
| Kernel context | PID, credentials, scheduling state, descriptors, signals, accounting | Lets the kernel manage, protect, observe, and eventually reclaim the execution |

![A program is passive input, while a process combines a loaded address space with kernel-managed execution state.](assets/program-to-process.svg){fig-alt="Diagram showing an executable file passing through a loader into a user virtual address space and kernel-maintained process state." width="96%"}

*Figure: original teaching diagram for this chapter. Click the image to inspect it at full size.*

The loader path in the figure should not be interpreted as eagerly copying the whole executable into RAM. Modern systems usually create virtual-memory mappings and bring pages into physical memory on demand. Loading establishes a valid future execution, while paging decides when individual bytes must become resident.

One executable can produce many processes. If three users run `grep`, the code pages may be physically shared because they are read-only, while each process still has distinct registers, stacks, credentials, descriptors, and writable data. Conversely, one process can replace its executable program through `execve()` without changing its PID. The process is the continuing kernel identity; the program image is replaceable content.

#### **Process Identity and Resource Ownership**

Unix-like systems identify a process with a **process ID**, or PID. `getpid()` returns the caller's PID and `getppid()` reports its parent. A PID is useful for observation and many control operations, but it is not a permanent global identity: PID values are reused, PID namespaces can give the same process different visible numbers, and a PID may refer to a different process after the original exits. Linux `pidfd` interfaces provide a race-resistant reference when software must retain a stable handle to a particular process.

A process also carries a collection of security and resource attributes:

- real, effective, and saved user and group credentials;
- a current working directory, root directory, and file-creation mask;
- resource limits and accounting counters;
- signal dispositions and pending signals;
- a table of file descriptors;
- references to an address space and executable image;
- parent-child relationships and an eventual exit status; and
- one or more schedulable threads.

"Ownership" does not mean every object is copied or exclusively held. Kernel objects are commonly reference-counted and shared. After `fork()`, parent and child initially refer to the same open-file descriptions, and copy-on-write lets their page tables refer to the same physical frames. The process boundary defines which references and permissions belong to each execution context, not a rule that all underlying data must be private.

This chapter initially treats a process as having one execution thread so that the control flow is easy to see. A later chapter separates **process resource ownership** from **thread execution state**. On Linux, this distinction also explains why many kernel details are represented per task rather than by one textbook-style process record.

### **The Process Image**

The **process image** is the complete state required to continue a process. In a narrow user-space sense it means the virtual address space. In a broader operating-system sense it includes CPU state and the kernel metadata that connects the process to protected resources.

#### **Virtual Address-Space Layout**

Each process observes a virtual address space rather than raw physical memory. A typical 64-bit Unix process may contain the following regions, although exact ordering and addresses vary by executable format, architecture, loader, security settings, and individual `mmap()` calls.

| Region | Typical contents | Common protection |
|---|---|---|
| Text | compiled machine instructions | read and execute |
| Read-only data | constants and immutable tables | read |
| Data | initialized writable globals | read and write |
| BSS | zero-initialized globals; represented compactly in the file | read and write |
| Heap | dynamic allocations traditionally expanded by `brk()` or supplied by the allocator | read and write |
| Memory mappings | shared libraries, mapped files, anonymous regions, thread stacks | mapping-dependent |
| User stack | call frames, local variables, initial arguments and environment | read and write, normally non-executable |
| Kernel-provided mappings | facilities such as Linux vDSO | mapping-dependent |

The layout is a set of mappings, not necessarily one dense block of allocated physical memory. An address can be valid but not resident; the first access can trigger a page fault that causes the kernel to allocate a zero-filled page or read data from a file. Unmapped addresses and forbidden accesses instead produce a fault that the process cannot normally recover from.

**Address-space layout randomization** changes mapping locations between executions, making fixed-address assumptions unreliable and raising the cost of many memory-corruption attacks. Position-independent executables and libraries cooperate with relocation mechanisms so code can run at those varying addresses. Later memory chapters examine page tables, the TLB, demand paging, and replacement in depth; here the important point is that a process image is a protected virtual view assembled from several kinds of backing objects.

#### **Registers, Stacks, and Kernel State**

Memory alone cannot identify an execution point. To pause and resume a process, the operating system must preserve its architectural state, including the program counter, stack pointer, general-purpose registers, status flags, and architecture-specific control state. Floating-point and vector registers may be included when used or managed with lazy/eager save policies chosen by the architecture and kernel.

The term **stack** can refer to two different structures:

- The **user stack** stores ordinary function frames, return addresses, local variables, and user-mode bookkeeping. Its pages belong to the process's virtual address space.
- A **kernel stack** is used while a thread executes kernel code after a system call, interrupt, or fault. User code cannot directly read or write it. It holds kernel call frames and may contain a saved trap frame for returning to user mode.

Keeping these stacks separate is a protection requirement. If the kernel trusted an arbitrary user stack while running privileged code, a process could modify kernel return state or make fault handling depend on untrusted memory. Entering the kernel therefore switches to controlled kernel state before normal kernel functions proceed.

When a process is not running, some saved state resides in memory owned by the kernel. When it is running, much of its current state is physically in CPU registers. The process image is thus partly concrete data structures and partly a logical promise: the kernel has enough information to reconstruct the next legal instruction.

#### **The Process Control Block**

Textbooks call the kernel's per-process record a **process control block** (PCB). It is an abstraction rather than a portable C structure. Real kernels distribute the information across architecture-specific context records, scheduling entities, memory descriptors, credential objects, descriptor tables, signal structures, and parent-child lists.

A conceptual PCB includes:

| Area | Representative information |
|---|---|
| Identity | PID, parent, process group, session |
| Execution | saved registers, kernel stack, current state |
| Scheduling | priority, policy, runtime, run-queue links, CPU affinity |
| Memory | reference to the address space and its mappings |
| Resources | descriptor table, working directory, limits |
| Protection | credentials, security labels, capabilities |
| Events | signal state, wait status, timers |
| Accounting | CPU time, start time, faults, I/O counters |

Linux's `task_struct` is often described as a PCB, but the mapping is not exact. Linux represents each schedulable thread as a task; threads in one process share selected structures such as the address space and descriptor table. Treating the PCB as a checklist of responsibilities is more useful than assuming one universal layout.

### **Process States and Transitions**

A process alternates between executing, being eligible to execute, and waiting for an event. States let the scheduler avoid wasting CPU time on work that cannot currently make progress.

#### **Ready, Running, Blocked, and Terminated**

The classic state model distinguishes:

- **New**: creation has begun, but the process is not yet eligible to run.
- **Ready**: the process can run as soon as a CPU is assigned.
- **Running**: instructions from this execution context are currently being executed on a CPU.
- **Blocked or waiting**: progress depends on an event such as input, a timer, a lock, or child termination.
- **Terminated**: execution has ended; some status metadata may remain until the parent reaps it.

![Five-state process model with admission, dispatch, preemption, event waiting, wakeup, and termination transitions.](assets/process-state-model.png){fig-alt="Five-state process diagram showing new, ready, running, blocked, and terminated states with labeled transitions." width="82%"}

*Figure source: [MrDrBob, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Process_state.svg), licensed under [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/). A local copy is used for reliable rendering.*

A state transition has a cause. A scheduler dispatch changes ready to running. Timer preemption can change running back to ready. A blocking `read()` can move a process from running to waiting, and device completion can make it ready again. Waking a process does **not** imply immediate execution; it joins an eligible queue and still competes for a CPU.

Linux exposes more detailed state codes in tools such as `ps` and `/proc/<pid>/stat`. Common codes include `R` for running or runnable, `S` for interruptible sleep, `D` for uninterruptible sleep, `T` for stopped or traced, and `Z` for zombie. These are implementation-oriented categories, so they should not be forced into a one-to-one mapping with every textbook diagram. In particular, Linux `R` combines the conceptual ready and running states.

The distinction between **interruptible** and **uninterruptible** sleep also matters. An interruptible sleeper can wake for a relevant signal, while a task in uninterruptible sleep is waiting in a kernel path that should not be abandoned at that point. A sustained `D` state often points to delayed I/O or a kernel subsystem, but the state alone is not a diagnosis.

#### **Parent, Child, Orphan, and Zombie Processes**

Unix process creation establishes a parent-child relation. The relation supports status collection and hierarchical management, but it does not mean a child executes inside the parent or that the parent must run first. After `fork()`, both are independently schedulable.

A child that calls `_exit()` or otherwise terminates becomes a **zombie** until its parent collects the termination status with `wait()` or `waitpid()`. A zombie does not retain the child's normal user memory or continue consuming CPU. It retains a small kernel record, including identity and status, because discarding that information immediately would race with the parent's later wait. Large numbers of unreaped zombies can still exhaust process identifiers or table capacity.

An **orphan** is a live child whose parent exits first. Unix-like systems reparent it to an implementation-defined system process, commonly the namespace's init process or a designated subreaper, which can eventually collect its status. Inside containers, PID 1 therefore has a practical responsibility to reap descendants.

The following Linux example deliberately leaves a zombie visible for a short interval. It is an observation tool, not a pattern for production code.

<details>
<summary>Show the annotated zombie observation program</summary>

```c
#include <stdio.h>
#include <stdlib.h>
#include <sys/types.h>
#include <sys/wait.h>
#include <unistd.h>

int main(void) {
    pid_t child = fork();
    if (child == -1) {
        perror("fork");
        return EXIT_FAILURE;
    }

    if (child == 0) {
        // _exit avoids flushing copied stdio buffers in the child.
        _exit(42);
    }

    printf("parent=%ld child=%ld\n", (long)getpid(), (long)child);
    printf("Inspect with: ps -o pid,ppid,stat,cmd -p %ld\n", (long)child);

    // During this delay the child has exited but has not been reaped.
    sleep(15);

    int status;
    if (waitpid(child, &status, 0) == -1) {
        perror("waitpid");
        return EXIT_FAILURE;
    }

    if (WIFEXITED(status)) {
        printf("collected exit status %d\n", WEXITSTATUS(status));
    }
    return EXIT_SUCCESS;
}
```

Compile and observe it with:

```bash
cc -Wall -Wextra -O2 zombie_demo.c -o zombie_demo
./zombie_demo
# In another terminal, run the ps command printed by the program.
```

</details>

### **Creating and Terminating Processes**

Unix deliberately separates **creating an execution branch** from **selecting the program that branch will run**. The separation makes descriptor redirection and pipeline construction remarkably composable, but it also creates an interval in which child setup must be handled carefully.

#### **fork, exec, wait, and exit**

`fork()` creates a child process by duplicating the caller's process context. It returns twice: the parent receives the child's PID, while the child receives zero. Both continue after the same call, and either one may run first.

`execve()` does something fundamentally different: it replaces the calling process's program image with a new one. On success it does not return. The PID is preserved, while the old address space, user stack, and instruction stream are replaced. This is why saying "exec creates a process" is incorrect.

`waitpid()` allows a parent to wait for and collect a child's state change. `exit()` performs user-space termination work such as flushing C standard-I/O streams and running registered handlers, then ultimately asks the kernel to terminate the process. `_exit()` enters kernel termination directly and is normally preferred in a post-`fork()` child when `exec` fails, because copied user-space buffers and handlers should not be run again.

![Animated process timeline showing fork splitting execution, child descriptor setup, exec replacing the child image, exit, and wait-based reaping.](assets/fork-exec-wait-animated.svg){fig-alt="Animated two-lane timeline for a parent and child showing fork, child setup, exec, exit, wait, and process reaping." width="96%"}

*Figure: original teaching animation for this chapter. The moving markers illustrate one legal history, not a guaranteed schedule; the complete sequence remains visible when animation is unavailable.*

The essential contracts are:

| Operation | Creates a new PID? | Replaces address space? | Important return behavior |
|---|---:|---:|---|
| `fork()` | yes | no; initially duplicates mappings using copy-on-write | returns in both parent and child |
| `execve()` | no | yes | returns only on failure |
| `waitpid()` | no | no | returns when the requested child state is available, or immediately with suitable options |
| `_exit()` | no | destroys the calling process's execution | never returns |

The Linux manual pages for [`fork(2)`](https://man7.org/linux/man-pages/man2/fork.2.html), [`execve(2)`](https://man7.org/linux/man-pages/man2/execve.2.html), and [`waitpid(2)`](https://man7.org/linux/man-pages/man2/wait.2.html) define the exact inherited and reset attributes. Several details are easy to miss:

- `fork()` duplicates descriptor-table entries, but those entries refer to the same open-file descriptions, so offsets and status flags can remain shared.
- `execve()` normally preserves open descriptors unless they have the close-on-exec flag.
- caught signal dispositions are reset by `execve()`, while many other process attributes have operation-specific rules.
- after `fork()` in a multithreaded process, the child contains only the calling thread; before `exec`, it should call only async-signal-safe operations unless the program has arranged stronger guarantees.
- `waitpid()` reports a packed status that must be interpreted with macros such as `WIFEXITED`, `WEXITSTATUS`, and `WIFSIGNALED`.

<details>
<summary>Show a minimal fork-exec-wait implementation</summary>

```c
#include <errno.h>
#include <stdio.h>
#include <stdlib.h>
#include <sys/types.h>
#include <sys/wait.h>
#include <unistd.h>

int main(void) {
    pid_t child = fork();
    if (child == -1) {
        perror("fork");
        return EXIT_FAILURE;
    }

    if (child == 0) {
        // Replace this child with grep. PATH lookup is provided by execlp().
        execlp("grep", "grep", "kernel", "input.txt", (char *)NULL);

        // Reached only when exec failed. errno still explains the failure.
        perror("execlp grep");
        _exit(127);
    }

    int status;
    while (waitpid(child, &status, 0) == -1) {
        if (errno == EINTR) {
            continue; // A signal interrupted wait; the child may still be live.
        }
        perror("waitpid");
        return EXIT_FAILURE;
    }

    if (WIFEXITED(status)) {
        printf("grep exited with status %d\n", WEXITSTATUS(status));
        return WEXITSTATUS(status);
    }
    if (WIFSIGNALED(status)) {
        printf("grep was terminated by signal %d\n", WTERMSIG(status));
        return 128 + WTERMSIG(status);
    }
    return EXIT_FAILURE;
}
```

</details>

This example is intentionally small, but its error path is semantically important. Exit status `127` is a shell convention for a command that could not be executed. More generally, a caller must decide how setup or execution failures are communicated; `fork()` succeeding does not imply that a later `exec()` will succeed.

#### **Process Spawning Beyond fork**

The fork-then-exec model is elegant, but it is not the only creation interface. Different systems expose different trade-offs.

| Interface | Main idea | Strength | Constraint or risk |
|---|---|---|---|
| `fork()` + `execve()` | clone the caller, customize child, replace image | maximum Unix composability | child setup is delicate in multithreaded programs; page-table creation still costs work |
| `posix_spawn()` | describe selected file and process actions in one spawn request | convenient and often cheaper or safer for ordinary command launch | supports a deliberate subset of arbitrary post-fork customization |
| `vfork()` + `execve()` | temporarily let the child share the parent's address-space context | can avoid duplication work on some implementations | parent is suspended; child must obey severe restrictions before exec or exit |
| Linux `clone()` family | select which execution resources are shared | foundation for threads, namespaces, and specialized runtimes | Linux-specific and easy to misuse directly |
| Windows `CreateProcess()` | create a process and load a specified executable as one operation | direct process-and-program creation model | different inheritance and setup conventions from Unix |

POSIX [`posix_spawn()`](https://man7.org/linux/man-pages/man3/posix_spawn.3.html) is particularly useful when a caller mainly needs descriptor actions, attributes, and an executable selection. Its implementation may use fork-like or vfork-like mechanisms, but applications program against the higher-level contract rather than assuming the internal method.

The important design lesson is not that one API is universally best. `fork()` exposes a general intermediate child state; spawn APIs constrain that state so an implementation can create common process shapes more efficiently and safely.

#### **Copy-on-Write as an Enabling Mechanism**

Naively copying every writable memory page during `fork()` would make process creation proportional to the parent's resident writable memory, even when the child immediately discards that memory with `execve()`. **Copy-on-write** (COW) avoids this eager copy.

![Copy-on-write sequence before fork, immediately after fork, and after the child writes a shared page.](assets/copy-on-write-fork.svg){fig-alt="Three-stage diagram showing parent pages before fork, shared read-only copy-on-write frames after fork, and a private child frame after a write fault." width="96%"}

*Figure: original teaching diagram for this chapter.*

The mechanism proceeds in stages:

1. Before `fork()`, the parent has writable virtual mappings to physical frames.
2. The kernel creates a child page-table structure whose relevant entries refer to the same frames.
3. Parent and child mappings are marked read-only for COW purposes, and the kernel records that the frames are shared.
4. Reads proceed without copying. If neither process writes before the child executes a new program, no content copy is needed.
5. A write causes a protection fault. The kernel allocates a private frame, copies the old page, remaps the writer's virtual page as writable, and resumes the faulting instruction.

COW delays data copying, but `fork()` is not free. The kernel still creates process metadata and page-table structures, manages reference counts, duplicates or shares resource references according to documented rules, and may invalidate or update translation state. The MIT xv6 material uses a COW fork exercise to expose exactly these page-table and fault-handling responsibilities: [xv6 book and labs](https://pdos.csail.mit.edu/6.1810/2025/xv6/book-riscv-rev5.pdf).

COW also preserves isolation. Once either process modifies a page, the change is private unless the mapping was deliberately created as shared memory. "Initially shares physical frames" is therefore not the same as "shares writable variables forever."

### **Loading and Linking a Program**

After a child has prepared descriptors and other inherited state, `execve()` asks the kernel to replace the process image. The executable format and loader define how file bytes become memory mappings and where the first user instruction comes from.

#### **Executable Layout and the Loader**

Linux commonly uses the **Executable and Linkable Format** (ELF). An ELF file contains headers plus data organized from two perspectives:

- **Sections** are mainly a linker's view: code, symbols, relocation information, string tables, debug data, and related categories.
- **Segments** are a loader's view: ranges that must be mapped with particular offsets, sizes, alignment, and permissions to construct the process image.

![ELF file layout showing the ELF header, program header table, sections, and section header table.](assets/elf-file-layout.png){fig-alt="Diagram of an ELF file with an ELF header, program header table, multiple sections, and a section header table." width="76%"}

*Figure source: [Suruena, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Elf-layout--en.svg), licensed under [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/). A local raster copy is used for consistent rendering.*

The generic ELF ABI describes program headers as the structures needed to create a process image: [ELF Program Loading](https://refspecs.linuxfoundation.org/elf/gabi4%2B/ch5.pheader.html). A simplified `execve()` path is:

1. Validate the executable, permissions, architecture, and format.
2. Build a new memory-map description rather than modifying the old image piecemeal.
3. Create mappings for loadable segments with appropriate read, write, and execute permissions.
4. Arrange zero-filled memory where a segment's memory size exceeds its file-backed size, which supports BSS.
5. If an interpreter is named, map the dynamic linker and transfer it enough metadata to load required shared objects.
6. Construct the initial user stack containing arguments, environment strings, and auxiliary metadata.
7. Install a new initial register state and commit the replacement.
8. Return to user mode at the selected entry point. The successful call never returns to the old program.

This high-level sequence hides implementation details but preserves the key transactional intuition: failure before commitment leaves the caller able to receive an error, while success destroys the old user-space execution path.

<details>
<summary>Show commands for inspecting an ELF executable</summary>

```bash
# Identify the file format, architecture, and linking style.
file /usr/bin/grep

# Inspect the ELF header and the loader-oriented program headers.
readelf -h /usr/bin/grep
readelf -l /usr/bin/grep

# Locate a requested program interpreter, if one exists.
readelf -l /usr/bin/grep | grep INTERP

# Inspect section metadata and dynamic dependencies without executing the file.
readelf -S /usr/bin/grep
readelf -d /usr/bin/grep

# Observe the mappings after a program is running.
grep -E 'grep|libc|ld-linux|\[stack\]|\[heap\]' /proc/<PID>/maps
```

Avoid running `ldd` on an untrusted executable. Depending on platform and file type, dependency inspection may involve the program's interpreter. Static inspection with `readelf` or `objdump` is the safer first choice.

</details>

#### **Static and Dynamic Linking**

**Linking** resolves references among separately compiled components. It can happen mostly before execution or continue during process startup and runtime.

| Property | Static linking | Dynamic linking |
|---|---|---|
| Library code | copied into the executable at link time | provided by shared objects mapped at runtime |
| Deployment | fewer runtime library dependencies | requires compatible loader and libraries |
| File and memory use | larger executable; code repeated across binaries | shared read-only library pages can reduce duplication |
| Updates | relink and redeploy the executable | compatible library updates can affect many programs |
| Startup work | less runtime symbol-resolution work | loader maps dependencies and performs relocations |
| Isolation and reproducibility | more self-contained, though kernel and external data still matter | behavior depends on the runtime library environment |

For a dynamically linked ELF program, the kernel maps the interpreter named by the executable, and that **dynamic linker** maps needed shared objects, performs relocations, initializes runtime structures, and transfers control toward the program entry path. Linux documents this role in [`ld.so(8)`](https://man7.org/linux/man-pages/man8/ld.so.8.html).

Position-independent code makes it possible to place shared objects at varying addresses. Symbol binding may occur eagerly at startup or lazily on first use, depending on executable, loader, and security settings. These choices trade startup latency against later surprises and can affect hardening. The simple statement "the kernel loads every library" is therefore misleading: the kernel establishes executable mappings and the interpreter path, while a user-space dynamic linker performs much of the dependency work under kernel-enforced mapping rules.

#### **Arguments, Environment, and Initial Stack**

A new program needs more than code. It must learn what command was requested and receive runtime metadata. Conceptually, the initial user stack contains data like:

```text
higher virtual addresses
+-------------------------------+
| argument and environment text |
+-------------------------------+
| auxiliary vector entries      |  AT_PAGESZ, AT_ENTRY, AT_EXECFN, ...
| NULL                           |
| envp[] pointers               |
| NULL                           |
| argv[] pointers               |
| argc                          |
+-------------------------------+  initial stack pointer
lower virtual addresses
```

The exact order, alignment, and metadata are ABI-specific. The C function `main(int argc, char **argv)` is not normally the first instruction executed. An entry routine such as `_start` receives the ABI-defined initial state, initializes the language runtime, arranges constructors and termination behavior, and then calls `main`.

The environment is inherited by default across `fork()` and supplied explicitly to `execve()` as `envp`. Because environment variables can influence library loading, locale, parsing, and application behavior, privileged programs must treat them as untrusted input rather than harmless configuration.

<details>
<summary>Show a C program that inspects startup data</summary>

```c
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <sys/auxv.h>
#include <unistd.h>

extern char **environ;

int main(int argc, char **argv) {
    printf("pid=%ld argc=%d\n", (long)getpid(), argc);

    for (int i = 0; i < argc; ++i) {
        printf("argv[%d] = %s\n", i, argv[i]);
    }

    // The auxiliary vector is supplied by the kernel/loader startup path.
    printf("page size = %lu\n", getauxval(AT_PAGESZ));
    printf("entry     = 0x%lx\n", getauxval(AT_ENTRY));

    const char *exec_name = (const char *)getauxval(AT_EXECFN);
    if (exec_name != NULL) {
        printf("exec file = %s\n", exec_name);
    }

    // Print only a bounded sample; a real environment can be large or secret.
    for (int i = 0; environ[i] != NULL && i < 3; ++i) {
        printf("env[%d] = %s\n", i, environ[i]);
    }
    return EXIT_SUCCESS;
}
```

```bash
cc -Wall -Wextra -O2 startup.c -o startup
DEMO_MODE=loader ./startup alpha "two words"
```

</details>

### **Context Switching**

Multiprogramming requires the CPU to stop executing one context and continue another. A **context switch** is the kernel-controlled transfer between execution contexts. It is related to, but not identical with, entering kernel mode.

#### **Saving and Restoring Execution State**

A process can leave the CPU because its time slice expires, it blocks, it yields, a higher-priority task becomes eligible, or the kernel makes another scheduling decision. A simplified process-to-process switch is:

![Context-switch path from process A through a kernel trap, saved state, scheduling, memory-context change, restoration, and process B.](assets/context-switch-path.svg){fig-alt="Flow diagram showing process A entering the kernel, saving state, scheduling process B, restoring B, and returning to user mode, with direct and indirect costs." width="96%"}

*Figure: original teaching diagram for this chapter.*

The path contains several conceptually distinct actions:

1. A trap, interrupt, fault, or blocking operation transfers control to a protected kernel entry path.
2. The kernel saves enough of the current execution state to resume process A later.
3. A's state and queue membership are updated: it may become ready, blocked, stopped, or terminated.
4. The scheduler selects process B according to policy and eligibility.
5. The kernel switches kernel execution context and, when A and B use different address spaces, selects B's memory-protection context.
6. B's saved state is restored, and a protected return resumes B in user mode or continues its pending kernel path.

A **mode switch** only changes privilege level. A system call can enter the kernel and return to the same process without selecting another process. Conversely, a context switch may occur while the kernel is already handling an event. Conflating the two makes performance counts and control-flow explanations inaccurate.

The exact saved register set depends on the architecture, calling path, and kernel. Not every register must be written to a PCB on every entry; assembly entry code and kernel stacks can preserve part of the state. The invariant is simply that the old context can later continue as if the intervening execution had not modified its private architectural state.

#### **Direct and Indirect Context-Switch Costs**

Context switching consumes no application-level work, but its cost is not one fixed number.

| Cost class | Examples | Why it varies |
|---|---|---|
| Direct software work | entry/return, register save/restore, scheduler queues, accounting | architecture, kernel path, enabled security features |
| Translation state | page-table context selection, TLB effects | process versus thread switch, PCID/ASID support, shared address space |
| Cache locality | displaced instructions and data, cache coherence | working-set size and whether the next task ran recently |
| Prediction state | branch predictor and front-end warmup | processor design and workload behavior |
| Topology effects | migration to another core or NUMA node | CPU affinity, load balancing, memory placement |

Switching between threads in one address space can avoid changing the page-table context, but it still changes registers, stacks, scheduler state, and potentially cache locality. Switching between processes may preserve more translation state than older descriptions suggest because modern CPUs tag translations with process-context identifiers. The correct conclusion is not "threads are free" or "every switch flushes the TLB," but that shared memory context can reduce one component of a workload-dependent cost.

Measurements must also distinguish **voluntary** context switches, often caused by blocking or yielding, from **involuntary** switches caused by preemption. A high count can indicate healthy I/O concurrency, lock contention, tiny time slices, or an overloaded machine; the count needs workload context.

<details>
<summary>Show Linux commands for observing switch activity</summary>

```bash
# Per-process voluntary and involuntary counts maintained by Linux.
grep -E 'voluntary_ctxt_switches|nonvoluntary_ctxt_switches' /proc/<PID>/status

# Repeated process-level context-switch reporting (sysstat package).
pidstat -w -p <PID> 1

# System-wide events while running one command (perf package).
perf stat -e context-switches,cpu-migrations,page-faults -- ./your_program

# Compare scheduling history with timestamps; requires suitable permission.
perf sched record -- ./your_program
perf sched timehist
```

These counters show symptoms, not causes. Pair them with CPU utilization, blocked time, system calls, lock profiles, and application-level latency before optimizing.

</details>

### **File Descriptors as Process Capabilities**

A Unix process accesses many kernel objects through small nonnegative integers called **file descriptors**. Standard input, output, and error conventionally occupy descriptors `0`, `1`, and `2`. Other values can refer to regular files, pipes, sockets, terminals, event sources, and device endpoints.

Calling a descriptor a **process capability** is useful in a practical sense: possessing an open descriptor gives the process authority to request operations supported by that handle, even if the original pathname later changes. Unix descriptors are not a complete formal capability system, because ambient namespaces and other authority can still exist, but descriptor passing and inheritance are central to least-authority designs.

#### **Descriptor Tables and Open File Descriptions**

The integer is an index in a per-process descriptor table. The table entry refers to a kernel **open file description**, which in turn refers to a file-system object, pipe endpoint, socket, or another kernel object.

![Per-process file-descriptor tables referencing system-wide open-file descriptions and inode objects.](assets/unix-file-table.png){fig-alt="Diagram showing two processes with file descriptor tables pointing to shared file-table entries and inode-table entries." width="82%"}

*Figure source: [Qwertyus, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:File_table_and_inode_table.svg), licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). A local copy is used for reliable rendering.*

Separating these layers explains several otherwise surprising behaviors:

| Layer | Representative state | Sharing behavior |
|---|---|---|
| Descriptor-table entry | descriptor flags such as `FD_CLOEXEC`, pointer to open-file description | copied as an entry by `fork()`; modified by `dup2()` or `close()` per process |
| Open-file description | current file offset, access mode, file status flags such as `O_APPEND` | can be shared by descriptors created with `dup()` and inherited across `fork()` |
| Underlying object | inode/file data, pipe buffer, socket state, device | can have many open-file descriptions or endpoints referring to it |

Two independent `open()` calls for the same path normally create two open-file descriptions with independent offsets. By contrast, `dup(fd)` creates another descriptor referring to the same open-file description, so the descriptors share an offset. After `fork()`, corresponding parent and child descriptors also refer to the same descriptions unless either process deliberately replaces or closes them.

<details>
<summary>Show a deterministic shared-offset demonstration</summary>

```c
#include <fcntl.h>
#include <stdio.h>
#include <stdlib.h>
#include <sys/types.h>
#include <sys/wait.h>
#include <unistd.h>

int main(void) {
    const char *path = "offset-demo.tmp";
    int fd = open(path, O_RDWR | O_CREAT | O_TRUNC, 0600);
    if (fd == -1) {
        perror("open");
        return EXIT_FAILURE;
    }

    if (write(fd, "ABCDE", 5) != 5 || lseek(fd, 0, SEEK_SET) == -1) {
        perror("prepare file");
        close(fd);
        return EXIT_FAILURE;
    }

    pid_t child = fork();
    if (child == -1) {
        perror("fork");
        close(fd);
        return EXIT_FAILURE;
    }

    if (child == 0) {
        char bytes[3] = {0};
        // This advances the shared open-file-description offset from 0 to 2.
        if (read(fd, bytes, 2) == 2) {
            dprintf(STDOUT_FILENO, "child read: %s\n", bytes); // AB
        }
        close(fd);
        _exit(0);
    }

    // Waiting makes the demonstration order deterministic.
    waitpid(child, NULL, 0);

    char bytes[3] = {0};
    if (read(fd, bytes, 2) == 2) {
        printf("parent read: %s\n", bytes); // CD, not AB
    }

    close(fd);
    unlink(path);
    return EXIT_SUCCESS;
}
```

</details>

The shared offset is a property of the open-file description, not global state in the inode and not a private integer stored directly in each descriptor-table slot.

#### **Redirection and Descriptor Inheritance**

Shell redirection works by changing the descriptor graph before `exec`. If a child opens `result.txt` and executes `dup2(output_fd, STDOUT_FILENO)`, descriptor `1` now refers to the same open-file description as `output_fd`. The program started by `exec` does not need to know a shell was involved; its normal writes to standard output reach the file.

Descriptor inheritance is powerful but must be controlled:

- close unused descriptors so resources and pipe endpoints do not remain alive accidentally;
- use `O_CLOEXEC`, `pipe2(O_CLOEXEC)`, or atomic close-on-exec variants when creating descriptors that should not cross an `exec` boundary;
- avoid a separate `open()` followed much later by `fcntl(F_SETFD)` in multithreaded launchers when another thread could fork in between;
- check `dup2()` and `close()` errors where failure affects correctness; and
- remember that successful `exec` preserves descriptors without `FD_CLOEXEC`, but replaces the program that can use them.

This inheritance model is why a pipeline can be assembled without modifying `cat` or `grep`. The shell passes authority by arranging descriptors, then those generic programs simply read `0` and write `1`.

### **How a Shell Builds the Running Pipeline**

The complete running example combines all previous mechanisms. The figure uses descriptor numbers `3`, `4`, and `5` as examples; actual numbers are whichever unused entries the kernel returns.

![Four-stage construction of the cat-to-grep pipeline: create kernel objects, fork children, redirect and exec, then close and reap.](assets/pipeline-process-build.svg){fig-alt="Four-column process diagram showing a shell creating a pipe and output file, forking cat and grep children, applying dup2 and exec, then closing descriptors and waiting." width="96%"}

*Figure: original teaching diagram for this chapter.*

The shell's causal sequence is:

1. Parse the pipeline and redirection before creating children.
2. Call `pipe()` to obtain read and write endpoints, then open `result.txt` with the requested create/truncate policy.
3. Fork the child that will become `cat`. In that child, connect standard output to the pipe's write end, close unused descriptors, and execute `cat input.txt`.
4. Fork the child that will become `grep`. Connect standard input to the pipe's read end, connect standard output to `result.txt`, close unused descriptors, and execute `grep kernel`.
5. In the shell, close both pipe endpoints and the output descriptor. The shell should not accidentally count as a writer or reader.
6. Schedule both children independently. The pipe buffers bytes and blocks a reader or writer when progress requires the other side.
7. When `cat` and every other process close the last write endpoint, `grep` can observe end-of-file after consuming buffered data.
8. Collect both exit statuses with `waitpid()` and derive the shell-visible pipeline status according to shell policy.

Closing descriptors is part of correctness, not housekeeping. If the parent retains a pipe write end, the kernel still sees a possible writer, so `grep` may wait forever for an EOF that cannot arrive while the shell is waiting for `grep`.

<details>
<summary>Show the complete annotated C pipeline</summary>

```c
#define _POSIX_C_SOURCE 200809L
#include <errno.h>
#include <fcntl.h>
#include <signal.h>
#include <stdio.h>
#include <stdlib.h>
#include <sys/types.h>
#include <sys/wait.h>
#include <unistd.h>

static void child_failure(const char *operation) {
    // perror is used here for clarity. Production multithreaded launchers
    // should keep the post-fork child path restricted to safe operations.
    perror(operation);
    _exit(127);
}

static int wait_for(pid_t child, const char *name) {
    int status;
    while (waitpid(child, &status, 0) == -1) {
        if (errno == EINTR) {
            continue;
        }
        perror("waitpid");
        return 1;
    }

    if (WIFEXITED(status)) {
        int code = WEXITSTATUS(status);
        fprintf(stderr, "%s exited with %d\n", name, code);
        return code;
    }
    if (WIFSIGNALED(status)) {
        int signal_number = WTERMSIG(status);
        fprintf(stderr, "%s terminated by signal %d\n", name, signal_number);
        return 128 + signal_number;
    }
    return 1;
}

int main(void) {
    int pipe_fd[2];
    if (pipe(pipe_fd) == -1) {
        perror("pipe");
        return EXIT_FAILURE;
    }

    int output_fd = open(
        "result.txt",
        O_WRONLY | O_CREAT | O_TRUNC,
        0644
    );
    if (output_fd == -1) {
        perror("open result.txt");
        close(pipe_fd[0]);
        close(pipe_fd[1]);
        return EXIT_FAILURE;
    }

    pid_t cat_pid = fork();
    if (cat_pid == -1) {
        perror("fork cat");
        close(pipe_fd[0]);
        close(pipe_fd[1]);
        close(output_fd);
        return EXIT_FAILURE;
    }

    if (cat_pid == 0) {
        // cat writes its normal stdout into the pipe.
        if (dup2(pipe_fd[1], STDOUT_FILENO) == -1) {
            child_failure("dup2 cat stdout");
        }

        close(pipe_fd[0]);
        close(pipe_fd[1]);
        close(output_fd);

        execlp("cat", "cat", "input.txt", (char *)NULL);
        child_failure("exec cat");
    }

    pid_t grep_pid = fork();
    if (grep_pid == -1) {
        perror("fork grep");
        close(pipe_fd[0]);
        close(pipe_fd[1]);
        close(output_fd);

        // Avoid leaving the already-created child unmanaged.
        kill(cat_pid, SIGTERM);
        waitpid(cat_pid, NULL, 0);
        return EXIT_FAILURE;
    }

    if (grep_pid == 0) {
        // grep reads from the pipe and writes matching lines to the file.
        if (dup2(pipe_fd[0], STDIN_FILENO) == -1) {
            child_failure("dup2 grep stdin");
        }
        if (dup2(output_fd, STDOUT_FILENO) == -1) {
            child_failure("dup2 grep stdout");
        }

        close(pipe_fd[0]);
        close(pipe_fd[1]);
        close(output_fd);

        execlp("grep", "grep", "kernel", (char *)NULL);
        child_failure("exec grep");
    }

    // The shell is neither a pipe reader nor writer for this foreground job.
    close(pipe_fd[0]);
    close(pipe_fd[1]);
    close(output_fd);

    int cat_status = wait_for(cat_pid, "cat");
    int grep_status = wait_for(grep_pid, "grep");

    // A simple shell commonly reports the last command's status. Real shells
    // may offer a pipefail policy that also considers earlier failures.
    (void)cat_status;
    return grep_status;
}
```

```bash
cc -Wall -Wextra -O2 pipeline.c -o pipeline
printf 'kernel entry\nuser process\nkernel scheduler\n' > input.txt
./pipeline
cat result.txt
```

</details>

The program shows the architecture of a shell pipeline but omits full job control. An interactive shell also manages process groups, terminal foreground ownership, signal forwarding, stopped jobs, and asynchronous status notifications. Those policies build on the same process and descriptor primitives.

### **Observing Processes in Linux**

Linux exposes process state through system calls, `/proc`, and tracing tools. Observation should answer a specific question rather than produce a wall of output.

| Question | Useful interface | What to inspect |
|---|---|---|
| Who created this process? | `ps`, `/proc/<PID>/status` | PID, PPID, process group, session |
| What is its current scheduler state? | `ps`, `/proc/<PID>/stat` | state code, CPU, priority, runtime |
| Which program image is installed? | `/proc/<PID>/exe`, command line | executable link and arguments |
| What memory image exists? | `/proc/<PID>/maps`, `pmap` | executable, libraries, heap, stack, mapped files |
| Which descriptors survived? | `/proc/<PID>/fd`, `lsof` | descriptor targets, pipes, sockets, files |
| Where is it blocked? | `strace`, `/proc/<PID>/wchan`, debugger | current system call or kernel wait point |
| What children and descendants exist? | `pstree`, `ps --forest` | process hierarchy and state |

<details>
<summary>Show a focused Linux observation workflow</summary>

```bash
# Start a pipeline in the background and retain the shell job PID.
sh -c 'cat input.txt | grep kernel > result.txt' &
job_pid=$!

# View hierarchy, state, process group, and wait channel.
ps -eo pid,ppid,pgid,sid,stat,psr,wchan:24,cmd --forest
pstree -ap "$job_pid"

# Inspect one selected process. Replace PID with a real value from ps/pstree.
pid=<PID>
readlink "/proc/$pid/exe"
tr '\0' ' ' < "/proc/$pid/cmdline"; echo
grep -E '^(Name|State|Pid|PPid|Threads|voluntary|nonvoluntary)' "/proc/$pid/status"
sed -n '1,20p' "/proc/$pid/maps"
ls -l "/proc/$pid/fd"

# Trace process creation, descriptor changes, exec, waiting, and exit.
strace -f -e trace=process,desc -o pipeline.trace \
  sh -c 'cat input.txt | grep kernel > result.txt'
less pipeline.trace
```

Short-lived commands may exit before manual inspection. For repeatable study, insert a controlled delay in a test program or rely on `strace -f`, which follows descendants and records the event sequence.

</details>

`/proc/<PID>/stat` is compact and machine-oriented; field positions and escaping details make casual parsing error-prone. Prefer documented parsers or targeted files such as `status` when possible. The official [`proc_pid_stat(5)`](https://man7.org/linux/man-pages/man5/proc_pid_stat.5.html) page defines Linux state characters and fields.

### **Comparison and Summary**

The main abstractions can now be separated cleanly:

| Concept | Stable question it answers | Can it exist without currently using a CPU? |
|---|---|---:|
| Program | What executable instructions and static data are available? | yes |
| Process | Which protected resources and identity belong to this execution? | yes |
| Thread | Which register stream and stack can the scheduler run? | yes |
| Address space | Which virtual addresses and permissions are visible? | yes |
| File descriptor | Which kernel object can this process access through this table entry? | yes |

Several common misconceptions are worth correcting explicitly:

- **"A process is a program."** A program is passive and reusable; a process adds live execution and kernel state.
- **"Running means the process exists."** Ready, blocked, stopped, and zombie processes are not running but still have kernel-visible state.
- **"fork copies every byte immediately."** Modern Unix implementations normally use copy-on-write, although process and page-table setup still costs work.
- **"exec creates another process."** Successful exec replaces the current process image and preserves the process identity.
- **"A zombie still owns all of its memory."** Most resources are gone; a small status record remains until collection.
- **"Entering the kernel always switches processes."** A mode switch can return to the same process without a context switch.
- **"A descriptor is the file."** It is a per-process table index that refers through an open-file description to an underlying object.
- **"Closing unused pipe ends is optional cleanup."** Endpoint lifetime determines end-of-file and can decide whether a pipeline terminates.

The end-to-end story for `cat input.txt | grep kernel > result.txt` is now precise. The shell creates kernel objects, forks execution branches, rewires descriptor tables, and executes new program images. The loader maps executable segments and builds initial stacks. The scheduler alternates runnable contexts while pipes block and wake them as data moves. Each process exits, retains waitable status briefly, and is finally reaped by the shell.

This chapter establishes the single-threaded process model. The next chapter, [Threads and Interprocess Communication](03-threads-and-interprocess-communication.html), asks what changes when several execution streams share one process and when separate processes must exchange data deliberately. Memory behavior is developed further in [Address Spaces, Page Tables, and the TLB](06-address-spaces-page-tables-and-tlb.html) and [Demand Paging and Memory Management](07-demand-paging-and-memory-management.html).
